# Luminary Pipelines Tutorial

This notebook walks through a complete end-to-end Pipelines workflow on the Luminary
platform using a parametric wing-body geometry.

**What you will build:** A 4-stage pipeline that:
1. Reads a parametric wing-body geometry (from a CSM file) and applies a Named Variable Set to
   produce a geometry variant
2. Validates the geometry variant before proceeding
3. Generates a mesh
4. Runs a RANS simulation

The pipeline is then invoked with a table of variants - each row in the table becomes a concurrent
run.

**What variants are generated:** The `wingbody.csm` file defines a fuselage-wing-flap
configuration with 6 design parameters: fuselage length and radius, wing semi-span, sweep angle,
and wing root position. In this tutorial you will sweep across combinations of these parameters -
for example varying wing span from 6 to 7 m and sweep angle from 25° to 30° - to explore how
planform geometry affects aerodynamic performance. Each variant is a fully independent geometry
rebuilt from the CSM, meshed, and simulated in parallel.

By the end of the tutorial you will know how to:
- Upload a parametric CSM geometry and prepare it for pipeline-driven sweeps
- Define a reusable pipeline with built-in and custom stages
- Invoke the pipeline with an arbitrary number of design variants in a single call
- Monitor job progress and access per-variant artifacts in the UI

> **Early Access:** Pipelines is an Early Access feature and may change as development continues.
Some setup steps in this notebook use internal SDK methods (prefixed with `_`) that will be
replaced with a public API in a future release.

## Prerequisites

Before running this notebook you need:

1. **A Luminary Cloud account** with API access
2. **The `luminarycloud` SDK** installed (`pip install luminarycloud`)
3. **The `wingbody.csm` file** - the parametric geometry used in this tutorial (can be found at
   https://github.com/luminarycloud/tutorials/blob/main/pipelines/wingbody.csm)

Everything else - project, geometry upload, Named Variable Sets, and Simulation Template - is
created by the cells below.

## Authentication

Set your API key as an environment variable before running this notebook:

```bash
export LC_API_KEY=your_api_key_here
```

You can create an API key in the Luminary Cloud UI under **Settings → API Keys**. If `LC_API_KEY`
is not set, you will be prompted to authenticate interactively on your first SDK call.

## Imports

In [ ]:
import uuid
import luminarycloud as lc
import luminarycloud.pipelines as pipelines
from luminarycloud.types import Vector3
from luminarycloud.params import simulation as sim_params
from luminarycloud.enum import *
from luminarycloud.params.enum import *
from luminarycloud import EntityIdentifier

print(f"luminarycloud SDK version: {lc.__version__}")

## Configuration

**Edit this cell** with your values. Everything else in the notebook uses these variables.

In [ ]:
# Project name - will be created if it does not already exist
PROJECT_NAME = "Wingbody Parametric Sweep (tutorial)"
PROJECT_DESC = "Wingbody parametric sweep via Pipelines"

# Design variants: each dict maps a CSM DESPMTR name to its value for that run.
# The wingbody.csm parameters are:
#   len   - fuselage length (default 8.0)
#   rad   - fuselage radius (default 0.5)
#   span  - wing semi-span (default 6.0)
#   sweep - wing sweep angle in degrees (default 30.0)
#   xroot - x-location of wing root (default 2.5)
#   yroot - y-location of wing root (default -0.2)
VARIANTS = [
    {"len": 8.0, "rad": 0.50, "span": 6.0, "sweep": 30.0, "xroot": 2.5, "yroot": -0.2},
    {"len": 8.0, "rad": 0.50, "span": 7.0, "sweep": 25.0, "xroot": 2.5, "yroot": -0.2},
    {"len": 8.0, "rad": 0.55, "span": 6.5, "sweep": 28.0, "xroot": 2.6, "yroot": -0.2},
    {"len": 9.0, "rad": 0.50, "span": 6.0, "sweep": 30.0, "xroot": 2.5, "yroot": -0.2},
    {"len": 9.0, "rad": 0.55, "span": 7.0, "sweep": 25.0, "xroot": 2.8, "yroot": -0.2},
]

---
## Part 1: Project & Geometry Setup

We create the project, upload the CSM file, and do a one-time preparation step so the geometry can
be driven by Named Variable Sets in the pipeline.

In [ ]:
# Create the project (or reuse if it already exists)
project = None
for p in lc.iterate_projects():
    if p.name == PROJECT_NAME:
        project = p
        print(f"Found existing project: {project.name} ({project.id})")
        break

if project is None:
    project = lc.create_project(PROJECT_NAME, PROJECT_DESC)
    print(f"Created project: {project.name} ({project.id})")

### Step 1a: Upload the CSM file

We upload `wingbody.csm` once as the base geometry. All pipeline variants will be derived from
this single base by applying different Named Variable Sets.

The code block below will download the CAD file from the tutorial Github:

In [ ]:
import requests

cad_url = "https://raw.githubusercontent.com/luminarycloud/tutorials/refs/heads/main/pipelines/wingbody.csm"
cad_file = "./wingbody.csm"
with open(cad_file, "wb") as f:
    f.write(requests.get(cad_url).content)
print("Base geometry downloaded")

In [ ]:
GEO_NAME = "Wingbody Base"

# Check if the base geometry already exists
base_geometry = None
for geo in project.list_geometries():
    if geo.name == GEO_NAME:
        base_geometry = geo
        print(f"Found existing base geometry: {base_geometry.name} ({base_geometry.id})")
        break

if base_geometry is None:
    print(f"Uploading {cad_file} ...")
    base_geometry = project.create_geometry(cad_file, name=GEO_NAME, scaling=1000, wait=True)
    print(f"Uploaded geometry: {base_geometry.name} ({base_geometry.id})")

### Step 1b: Prepare geometry for parametric sweep

This one-time step binds each CSM parameter to a Named Variable expression so the pipeline
can substitute different values per run.

In [ ]:
nvs = base_geometry.bind_import_to_named_variable_set()

print("Geometry prepared for parametric sweep. Variables:")
for var_name, value in nvs._get_named_variables().items():
    print(f"  {var_name} = {value}")

In [ ]:
# Check if a farfield has already been added to avoid duplicates on re-runs
existing_features = base_geometry._list_features()
has_farfield = any(feat.HasField("farfield") for feat in existing_features)

if has_farfield:
    print("Farfield already present - skipping.")
else:
    farfield_shape = lc.params.geometry.HalfSphere(
        center=Vector3(0.0, 0.0, 0.0),
        radius=100.0,
        normal=Vector3(0.0, 0.0, 1.0),  # flat side faces downward
    )
    base_geometry.add_farfield(farfield_shape)
    print("Farfield added to base geometry.")

BASE_GEO_ID = base_geometry.id
print(f"Base geometry ID: {BASE_GEO_ID}")

### Step 1c: Create the Simulation Template

The simulation template is created programmatically - no UI step required.

In [ ]:
# Tag covering wingbody wall surfaces on pipeline-generated variants.
# Variants use the geometry worker's NVS replay path → single-product STEP → "7.7 1".
BODY_TAG = "Open CASCADE STEP translator 7.7 1"

SIM_TEMPLATE_NAME = "Wingbody RANS"

# Fixed IDs matching the reference template - kept stable so re-runs are idempotent.
MATERIAL_ID = "b1bqy2pzpm56q5qjde1104c8hk5nb33k"
PHYSICS_ID = "v6zjg5j6xegy5vn0r9pfbdc7s1otcz81"


def _rand_id():
    return str(uuid.uuid4()).replace("-", "")[:32]


# Create or reuse the simulation template
existing_sts = {st.name: st for st in project.list_simulation_templates()}
if SIM_TEMPLATE_NAME in existing_sts:
    sim_template = existing_sts[SIM_TEMPLATE_NAME]
    print(f"Reusing existing sim template: {sim_template.name} ({sim_template.id})")
else:
    sim_template = project.create_simulation_template(name=SIM_TEMPLATE_NAME)
    print(f"Created sim template: {sim_template.name} ({sim_template.id})")

# --- Entity relationships ---
obj = lc.SimulationParam()
obj.entity_relationships = sim_params.EntityRelationships()

vmr = sim_params.entity_relationships.VolumeMaterialRelationship()
vmr.material_identifier = EntityIdentifier(id=MATERIAL_ID)
vmr.volume_identifier = EntityIdentifier(id="0")
obj.entity_relationships.volume_material_relationship = [vmr]

vpr = sim_params.entity_relationships.VolumePhysicsRelationship()
vpr.physics_identifier = EntityIdentifier(id=PHYSICS_ID)
vpr.volume_identifier = EntityIdentifier(id="0")
obj.entity_relationships.volume_physics_relationship = [vpr]

# --- Material: Standard Air ---
mat = sim_params.MaterialEntity()
mat.material_identifier = EntityIdentifier(id=MATERIAL_ID, name="Fluid 1")
mat.fluid = sim_params.material.MaterialFluid()
mat.fluid.preset = MaterialFluidPreset.STANDARD_AIR
obj.materials = [mat]

# --- Physics: RANS / Spalart-Allmaras ---
phys = sim_params.Physics()
phys.physics_identifier = EntityIdentifier(id=PHYSICS_ID, name="Fluid Flow 1")
phys.fluid = sim_params.physics.Fluid()

# Spalart-Allmaras turbulence model
phys.fluid.turbulence = sim_params.physics.fluid.turbulence.SpalartAllmaras()

# Farfield BC - Mach 0.2 along +x
farfield_bc = sim_params.physics.fluid.boundary_conditions.Farfield()
farfield_bc.id = _rand_id()
farfield_bc.surfaces = ["lcTag/tagContainer/tag/Farfield"]
farfield_bc.momentum = FarfieldMomentum.FARFIELD_MACH_NUMBER
farfield_bc.mach_number = 0.2
farfield_bc.direction = Vector3(x=1.0, y=0.0, z=0.0)
farfield_bc.temperature = 288.15
farfield_bc.pressure = 101325.0

# Symmetry BC - flat face of the HalfSphere farfield
symmetry_bc = sim_params.physics.fluid.boundary_conditions.Symmetry()
symmetry_bc.id = _rand_id()
symmetry_bc.surfaces = ["lcTag/tagContainer/tag/Symmetry Plane"]

# Wall BC - wingbody surface
wall_bc = sim_params.physics.fluid.boundary_conditions.Wall()
wall_bc.id = _rand_id()
wall_bc.surfaces = [f"lcTag/tagContainer/tag/{BODY_TAG}"]
wall_bc.momentum = sim_params.physics.fluid.boundary_conditions.wall.momentum.NoSlip()

phys.fluid.boundary_conditions = [farfield_bc, symmetry_bc, wall_bc]

# Initialization from farfield values
init = sim_params.physics.fluid.initialization.FluidFarfieldValues()
init.turbulence = sim_params.physics.fluid.initialization.TurbulenceInitialization()
init.turbulence.spalart_allmaras = TurbulentVariableInitializationTypeSa.INIT_FARFIELD_VALUES_SA
phys.fluid.initialization = init

obj.physics = [phys]

# --- Reference values ---
obj.reference_values = lc.reference_values.ReferenceValues()
obj.reference_values.reference_value_type = ReferenceValuesType.PRESCRIBE_VALUES
obj.reference_values.p_ref = 101325.0
obj.reference_values.t_ref = 288.15

sim_template.update(parameters=obj)
sim_template.update_general_stopping_conditions(2000, 1.0, 5, False)

SIM_TEMPLATE_ID = sim_template.id
print(f"\nSim template ready: {SIM_TEMPLATE_ID}")
print(f"  Farfield BC   → Farfield        (Mach 0.2, +x)")
print(f"  Symmetry BC   → Symmetry Plane")
print(f"  Wall BC       → {BODY_TAG}")

---
## Part 2: Create Named Variable Sets

Each Named Variable Set (NVS) corresponds to one pipeline run. The variable names must match the
parameter names in your CSM file - for `wingbody.csm` those are `len`, `rad`, `span`, `sweep`,
`xroot`, and `yroot`.

In [ ]:
# Build a display name from the variant's key parameters
def variant_name(i, params):
    return (
        f"variant_{i:02d}_"
        f"span{params['span']}_"
        f"sweep{params['sweep']}_"
        f"len{params['len']}"
    )


# Create one NVS per variant (skip if it already exists)
existing_nvs = {nvs.name: nvs for nvs in project.list_named_variable_sets()}

nv_sets: list[lc.NamedVariableSet] = []
for i, params in enumerate(VARIANTS):
    name = variant_name(i, params)
    if name in existing_nvs:
        print(f"  Reusing existing NVS: {name}")
        nv_sets.append(existing_nvs[name])
    else:
        nvs = project.create_named_variable_set(name, params)
        print(f"  Created NVS: {name} ({nvs.id})")
        nv_sets.append(nvs)

print(f"\nTotal Named Variable Sets: {len(nv_sets)}")

---
## Part 3: Define the Pipeline

A Pipeline is a directed acyclic graph (DAG) of stages. We define it once on the platform and can
invoke it as many times as needed with different argument tables.

Our pipeline has four stages:

```
Create Geometry  →  Validate Geometry  →  Create Mesh  →  Create Simulation
```

### Pipeline Parameters

`PipelineParameter` objects are typed placeholders. They appear as column headers in the
`PipelineArgs` table and get substituted at runtime for each run.

In [ ]:
pp_geo_id = pipelines.StringPipelineParameter("geo_id")
pp_expected_vols = pipelines.IntPipelineParameter("expected_num_volumes")
pp_sim_template_id = pipelines.StringPipelineParameter("sim_template_id")
pp_variant_name = pipelines.StringPipelineParameter("variant_name")

### Stage 1: CreateGeometry

`CreateGeometry` copies the base geometry and applies the run's Named Variable Set to produce the
shape variant for that run. Each run gets its own independent copy, so variants can run
concurrently without interfering with each other.

In [ ]:
create_geo = pipelines.CreateGeometry(
    base_geometry_id=pp_geo_id,
    geo_name=pp_variant_name,
)

### Stage 2: Geometry Validation (RunScript)

The `@pipelines.stage` decorator turns a Python function into a custom stage (`RunScript`). We use
it here to validate the generated geometry before committing compute resources to meshing and
simulation.

Key rules for `RunScript` functions:
- **All imports must be inside the function body** - the function is serialised and executed in an
  isolated environment.
- **Raise `pipelines.StopRun`** to abort a single run gracefully without failing the whole job.
- **Short timeout** - Do not wait for long-running operations inside a RunScript. Functions that
  take too long to complete will be killed. (The limit is currently 5 minutes, but that is subject
  to change without notice.)

In [ ]:
@pipelines.stage(
    stage_name="Validate Geometry",
    inputs={"geometry": create_geo.outputs.geometry},
    outputs={"geometry": pipelines.PipelineOutputGeometry},
    params={"expected_num_volumes": pp_expected_vols},
)
def validate_geo(geometry: lc.Geometry, expected_num_volumes: int):
    # All imports must be inside the function body
    import luminarycloud.pipelines as pipelines

    print(f"Validating geometry: {geometry.id}")
    _surfaces, volumes = geometry.list_entities()

    if len(volumes) != expected_num_volumes:
        # StopRun ends this run cleanly - other runs in the job are not affected
        raise pipelines.StopRun(
            f"Expected {expected_num_volumes} volume(s), got {len(volumes)} - "
            f"skipping mesh and simulation for geometry {geometry.id}"
        )

    print(f"Geometry {geometry.id} is valid ({len(volumes)} volume(s))")
    return {"geometry": geometry}

### Stages 3 & 4: Mesh and Simulate

Stage outputs are wired as inputs to downstream stages using dot notation (e.g.
`validate_geo.outputs.geometry`). This is how the pipeline DAG is expressed in code.

In [ ]:
create_mesh = pipelines.CreateMesh(
    geometry=validate_geo.outputs.geometry,
    mesh_name=pp_variant_name,
    # target_cv_count=500_000,  # uncomment to request a specific mesh resolution. Default is a minimal mesh.
)

create_sim = pipelines.CreateSimulation(
    sim_template_id=pp_sim_template_id,
    mesh=create_mesh.outputs.mesh,
    sim_name=pp_variant_name,
)

### Create the Pipeline

`create_pipeline` registers the pipeline on the Luminary platform. This only needs to happen
once - the returned `pipeline` object can be invoked many times with different argument tables.

In [ ]:
PIPELINE_NAME = "Wingbody CSM sweep - validate, mesh, RANS"

# Check if a pipeline with this name already exists
pipeline = None
for p in pipelines.iterate_pipelines():
    if p.name == PIPELINE_NAME:
        pipeline = p
        print(f"Found existing pipeline: {pipeline.name} ({pipeline.id})")
        break

if pipeline is None:
    pipeline = pipelines.create_pipeline(
        name=PIPELINE_NAME,
        description="Created from the Pipelines Tutorial Notebook",
        stages=[create_geo, validate_geo, create_mesh, create_sim],
    )
    print(f"Created pipeline: {pipeline.name} ({pipeline.id})")

---
## Part 4: Invoke the Pipeline

We invoke the pipeline by supplying a `PipelineArgs` table. Each row becomes one
**PipelineJobRun**. All runs execute concurrently.

The `PP_NAMED_VARIABLE_SET_ID` column is special: it tells `CreateGeometry` which NVS to apply to
the base geometry for each run. This is what drives the geometry variation.

In [ ]:
# EXPECTED_NUM_VOLUMES: wingbody.csm produces one body (fuselage + wing + flap unioned),
# and add_farfield wraps that in a farfield, giving 1 fluid-domain volume.
# Adjust this if your geometry produces a different topology.
EXPECTED_NUM_VOLUMES = 1

arg_cols = [
    pipelines.PP_NAMED_VARIABLE_SET_ID,
    pp_variant_name,
    pp_geo_id,
    pp_sim_template_id,
    pp_expected_vols,
]

# Every run uses the same base geometry, sim template, and expected number of volumes. The NVS
# varies, and we'll name the mesh and simulation the same as the NVS to make it easier to know at a
# glance where they came from.
arg_rows = [
    [nvs.id, nvs.name, BASE_GEO_ID, SIM_TEMPLATE_ID, EXPECTED_NUM_VOLUMES] for nvs in nv_sets
]

args = pipelines.PipelineArgs(arg_cols, arg_rows)

print(f"PipelineArgs table ({len(arg_rows)} runs):")
args.print_as_table()

In [ ]:
JOB_NAME = f"Wingbody sweep - {len(VARIANTS)} variants"

pipeline_job = pipeline.invoke(
    args, job_name=JOB_NAME, description="Created from the Pipelines Tutorial Notebook"
)

print(f"Pipeline job created: {pipeline_job.id}")
print()
print("The job is now running asynchronously on the Luminary platform.")
print("Monitor progress in the Pipelines dashboard in the UI:")
print(pipeline_job.url)
print("Or use the polling cell below.")

---
## Part 5: Monitoring

The best place to monitor a pipeline job is the **Luminary UI**:

| View | What you see |
|------|--------------|
| **Pipelines dashboard** | All pipelines and recent jobs |
| **Job view** | Status of every run and task; filter by status to isolate failures |
| **Run details** | Per-run arguments, links to all generated artifacts (geometry variants, meshes, simulations), and error messages for failed tasks |
| **Pipeline graph** | Visual DAG of the pipeline structure |

You can also poll programmatically from this notebook:

In [ ]:
# Optional: block this notebook until the job reaches a terminal state.
# Remove or skip this cell if you want the notebook to return immediately.
#
# by default, waits indefinitely. pass a `timeout_seconds` value to cap the wait.

print(f"Waiting for job {pipeline_job.id} to finish...")
final_status = pipeline_job.wait()
print(f"Job finished with status: {final_status}")

### Programmatic job control

You can pause, resume, or cancel a running job, and tune per-stage concurrency:

In [ ]:
# Retrieve the job at any time using its ID
job = pipelines.get_pipeline_job(pipeline_job.id)

# Uncomment one of the following to control the job:
# job.pause()                                  # pause all pending tasks
# job.resume()                                 # resume after a pause
# job.cancel()                                 # cancel all pending tasks
# job.set_concurrency_limits({create_sim.id: 2})  # limit simulations to 2 at a time

print(f"Job ID:   {job.id}")

---
## Next Steps

Now that you have a working parametric pipeline, here are some directions to explore:

**Larger sweeps**
- Extend the `VARIANTS` list in the Configuration cell - the pipeline definition stays the same
- Use `random.uniform` or a Latin Hypercube sampler to generate hundreds of variants programmatically

**Custom meshing**
- Replace the `Mesh` stage with a `RunScript` + `WaitForMesh` pair to control boundary layer parameters per variant

**Reusing the pipeline**
- Once the pipeline exists on the platform you can skip Part 3 entirely and go straight to Part 4 with a new set of variants:

```python
pipeline = next(p for p in pipelines.iterate_pipelines() if p.name == PIPELINE_NAME)
pipeline.invoke(new_args, job_name="Second sweep")
```

See the [Pipelines documentation](https://app.luminarycloud.com/docs/api/pipelines.html) for the full API reference.